# 05b · MLSD, comparação com Lira et al. (2024) e efeito do vazamento — tabelas da dissertação

**Sessão P1 · CPU** (Colab sem GPU, ou localmente). Lê `resultados/` (02, 03, 04, 04b), confere a integridade, recalcula as métricas das predições salvas e gera:

1. **MLSD:** média ± dp entre dobras e a **mesma regra de decisão pré-registrada** de entrada/saída (KAN (d) × melhor MLP: ΔAUC > 0 em ≥ 4/5 dobras **e** IC 95 % do bootstrap de casos excluindo 0) → `tabela_mlsd.{csv,tex}`, `decisao_mlsd.json`;
2. **TransKAN × Swin-B e × MaxViT-T** nas duas tarefas (ΔAUC pareado + bootstrap de casos) → `comparacao_transkan_transformers.json`;
3. **formato das Tabelas 4 e 5 de Lira** (Acc; P, R, F1, AUC e especificidade em macro/micro/ponderada), recalculado das predições → `tabela_mmw_{es,dist}.{csv,tex}`;
4. **Lira (2024) × mesmo protocolo**, por backbone → `tabela_comparacao_2024_{es,dist}.{csv,tex}` (inclui a "AUC de classes" — a forma de Lira — ao lado da AUC de probabilidades);
5. **vazamento:** hold-out de Lira × dobras agrupadas, mesma receita → `tabela_vazamento.{csv,tex}`, `vazamento_vs_agrupado.png`;
6. `roc_mlsd_cabecas.png` e `frases_dissertacao.md` (PT e EN, com o que pode e o que não pode ser afirmado).

Entrada/saída continua decidida no 05 (este notebook repete a mesma conta, com a mesma semente do bootstrap, só para juntar as duas tarefas).

In [ ]:
# ==== Configuração ====
NO_COLAB = True    # False para rodar localmente (aponte os caminhos abaixo para cópias locais)
DIR_DRIVE = "/content/drive/MyDrive/Colab Notebooks/tcc-final"
DIR_CODIGO = DIR_DRIVE                      # protocolo_comum.py e configs_mlsd.py (localmente: notebooks/)
DIR_RESULTADOS = f"{DIR_DRIVE}/resultados"  # localmente: resultados/
DIR_DADOS = f"{DIR_DRIVE}/dados"            # folds_entrada_saida.csv, holdout_lira2024.csv, lira2024_tabelas.csv
SHA_PROTOCOLO_ESPERADO = "b7cd9baca53edd96c855cca10914fc258efa8ddfd0937cc0d3e9572de5ef9e13"
SHA_PLUGIN_ESPERADO = "e2138640a83ef7587200ad6e051262b6a49e5b60027f6b4a61407fd1f61889f7"
N_BOOT = 2000

In [ ]:
if NO_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
                             balanced_accuracy_score, confusion_matrix, roc_curve)
sys.path.insert(0, DIR_CODIGO)
import protocolo_comum as pc
assert pc.sha256_protocolo() == SHA_PROTOCOLO_ESPERADO, "protocolo_comum.py diferente do publicado"
assert pc.sha256_arquivo(f"{DIR_CODIGO}/configs_mlsd.py") == SHA_PLUGIN_ESPERADO, "configs_mlsd.py diferente do publicado"
pc.carregar_plugin(f"{DIR_CODIGO}/configs_mlsd.py")
cm = sys.modules["configs_mlsd"]
R = Path(DIR_RESULTADOS)
E1, E2, H, B = pc.CONFIGS_EXP1, pc.CONFIGS_EXP2, cm.CONFIGS_CABECA, cm.CONFIGS_BASELINES
CAB = {t: {"es": E1, "dist": H}[t] for t in ("es", "dist")}          # cabeças (a)–(e) por tarefa
BAS = {"es": {**E2, "resnet152": E1["a"]}, "dist": {**B, "resnet152": H["a"]}}   # backbones (cabeça linear)
NOMES_BB = {"resnet152": "ResNet152", "alexnet": "AlexNet", "mobilenet_v2": "MobileNetV2", "mnasnet1_3": "MNASNet1.3",
            "shufflenet_v2_x0_5": "ShuffleNetV2 x0.5", "squeezenet1_0": "SqueezeNet1.0", "densenet169": "DenseNet169",
            "maxvit_t": "MaxViT-T", "swin_b": "Swin-B", "efficientnet_v2_l": "EfficientNetV2-L", "vgg16": "VGG16",
            "regnet_y_16gf": "RegNetY-16GF"}
NOMES_CAB = {"a": "ResNet152 + Linear (a)", "b": "ResNet152 + MLP-128 (b)", "c": "ResNet152 + MLP-iso (c)",
             "d": "ResNet152 + KAN (d) = TransKAN", "e": "ResNet152 + KAN-8 (e)"}
NOMES = {}
for t in ("es", "dist"):
    NOMES.update({CAB[t][k]: n for k, n in NOMES_CAB.items()})
    NOMES.update({BAS[t][b]: n for b, n in NOMES_BB.items() if b != "resnet152"})
EXPS = {"es": ["ablacao_cabeca", "baselines"], "dist": ["mlsd_cabeca", "mlsd_baselines"]}
ROTULO = {"es": "entrada/saída", "dist": "MLSD (distância)"}

def ler(exps):
    partes = [pc.ler_resultados(R, e) for e in exps]
    partes = [p for p in partes if not p.empty]
    if not partes:
        return pd.DataFrame()
    r = pd.concat(partes, ignore_index=True)
    return r[~r["teste_rapido"].astype(bool)].copy()

runs = {t: ler(EXPS[t]) for t in EXPS}
runs_v = ler([f"vazamento_lira_p{p}" for p in cm.SEMENTES_PARTICAO])
dobras = pc.ler_dobras(Path(DIR_DADOS) / pc.ARQUIVO_DOBRAS)
caso_real = dobras.set_index("rel_path")["case_id"]
lira = pd.read_csv(Path(DIR_DADOS) / "lira2024_tabelas.csv")
for t in runs:
    print(ROTULO[t], ":", len(runs[t]), "unidades")
    if not runs[t].empty:
        display(runs[t].groupby(["experimento", "config", "semente"]).size().rename("dobras"))
print("vazamento:", len(runs_v), "unidades")

## 1. Integridade (hashes, precisão, plugin, métricas recalculadas das predições)

In [ ]:
def prob_cols(t):
    return [f"p_{t}_{c}" for c in range(pc.TAREFAS[t].n_classes)]

def auc_tarefa(y, P):
    if P.shape[1] == 2:
        return roc_auc_score(y, P[:, 1])
    return roc_auc_score(y, P, multi_class="ovr", average="macro", labels=list(range(P.shape[1])))

def tarefa_do_run(r):
    return next(iter(json.loads(r["cabecas"])))

problemas, avisos = [], []
todos = pd.concat([x for x in [*runs.values(), runs_v] if not x.empty], ignore_index=True)
for col in ("protocolo_sha256", "dobras_sha256", "precisao"):
    vals = todos[col].dropna().unique()
    if len(vals) != 1:
        problemas.append(f"{col} não é único: {vals}")
if todos["protocolo_sha256"].iloc[0] != SHA_PROTOCOLO_ESPERADO:
    problemas.append("SHA do protocolo diferente do publicado")
todos["reserva"] = ~todos["gpu"].fillna("").str.contains("L4")
for t in runs:
    if not runs[t].empty:
        runs[t]["reserva"] = ~runs[t]["gpu"].fillna("").str.contains("L4")
if todos["reserva"].any():
    print("† unidades fora da L4 (config × semente: nº de dobras):")
    display(todos.loc[todos.reserva].groupby(["experimento", "config", "semente"]).size())
precisa_plugin = todos["config"].str.startswith("mlsd__") | todos["experimento"].str.startswith("vazamento_lira")
for _, r in todos[precisa_plugin].iterrows():
    if json.loads(r["plugins"]).get("configs_mlsd") != SHA_PLUGIN_ESPERADO:
        problemas.append(f"{r['experimento']}/{r['config']} d{r['dobra']}: rodou com outro configs_mlsd.py")

oof = {"es": [], "dist": []}
oof_v = []
for _, r in todos.iterrows():
    t = tarefa_do_run(r)
    tar = pc.TAREFAS[t]
    p = pd.read_csv(Path(r["caminho"]) / "pred_teste.csv")
    v = p[p[tar.coluna] >= 0]
    y, P = v[tar.coluna].to_numpy(), v[prob_cols(t)].to_numpy()
    pred = P.argmax(1)
    rec = {f"{t}_acc": accuracy_score(y, pred), f"{t}_auc": auc_tarefa(y, P),
           f"{t}_rec_macro": recall_score(y, pred, average="macro"),
           f"{t}_prec_macro": precision_score(y, pred, average="macro", zero_division=0),
           f"{t}_f1_macro": f1_score(y, pred, average="macro"), f"{t}_acc_balanceada": balanced_accuracy_score(y, pred)}
    for k, val in rec.items():
        if abs(val - r[k]) > 1e-9:
            problemas.append(f"{r['config']} s{r['semente']} d{r['dobra']}: {k} recalculado {val} ≠ {r[k]}")
    if abs(r[f"{t}_auc"] - r[f"{t}_rec_macro"]) < 1e-12:
        avisos.append(f"{r['config']} s{r['semente']} d{r['dobra']}: AUC = revocação macro (modelo degenerado?)")
    p = p.assign(config=r["config"], semente=r["semente"], dobra=r["dobra"], experimento=r["experimento"], tarefa=t)
    if r["experimento"].startswith("vazamento_lira"):
        m = json.loads((Path(r["caminho"]) / "metricas.json").read_text())
        if not m["dados"].get("holdout_lira"):
            problemas.append(f"{r['experimento']}/{r['config']}: unidade de vazamento sem a marca holdout_lira")
        oof_v.append(p)
    else:
        oof[t].append(p)
oof = {t: pd.concat(v, ignore_index=True) if v else pd.DataFrame() for t, v in oof.items()}
oof_v = pd.concat(oof_v, ignore_index=True) if oof_v else pd.DataFrame()
if not oof["dist"].empty:
    saidas_treino = 0  # as saídas nunca treinam (checado no plugin); aqui só confere que não entram na métrica
    assert (oof["dist"].loc[oof["dist"]["y_dist"] < 0, "y_es"] == 1).all()
print("Problemas:", problemas or "nenhum")
print("Avisos:", avisos or "nenhum")
assert not problemas

## 2. Média ± dp entre dobras (cada dobra = média das sementes disponíveis)

In [ ]:
def resumir(t):
    r = runs[t]
    if r.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.Series(dtype=object)
    metr = [f"{t}_{m}" for m in ("acc", "prec_macro", "rec_macro", "f1_macro", "acc_balanceada", "auc")]
    por_dobra = r.groupby(["config", "dobra"])[metr + ["treino_min"]].mean().reset_index()
    sementes = r.groupby("config")["semente"].apply(lambda s: sorted(s.unique())).rename("sementes")
    agg = por_dobra.groupby("config")[metr + ["treino_min"]].agg(["mean", "std", "count"])
    agg.columns = [f"{a}_{b}" for a, b in agg.columns]
    dp_sem = r.groupby(["config", "semente"])[f"{t}_auc"].mean().groupby("config").std(ddof=1).rename("dp_auc_entre_sementes")
    info = r.groupby("config")[["params_cabeca", "params_total", "reserva"]].agg(
        {"params_cabeca": "first", "params_total": "first", "reserva": "any"})
    return agg.join(sementes).join(dp_sem).join(info), por_dobra, sementes

RES, POR_DOBRA, SEMENTES = {}, {}, {}
for t in ("es", "dist"):
    RES[t], POR_DOBRA[t], SEMENTES[t] = resumir(t)
    if not RES[t].empty:
        print(ROTULO[t]); display(RES[t].round(4))
# recall por classe da MLSD (OOF por dobra, média das sementes)
def recall_por_classe(t, cfg):
    o = oof[t][(oof[t].config == cfg) & (oof[t][pc.TAREFAS[t].coluna] >= 0)]
    linhas = []
    for (s, d), g in o.groupby(["semente", "dobra"]):
        y = g[pc.TAREFAS[t].coluna].to_numpy(); pred = g[prob_cols(t)].to_numpy().argmax(1)
        linhas.append({"semente": s, "dobra": d, **{f"rec_{n}": v for n, v in zip(pc.TAREFAS[t].nomes_classes,
                       recall_score(y, pred, average=None, labels=list(range(len(pc.TAREFAS[t].nomes_classes))), zero_division=0))}})
    x = pd.DataFrame(linhas).groupby("dobra").mean().drop(columns="semente")
    return x.mean(), x.std(ddof=1)
if not RES["dist"].empty:
    rec_dist = {cfg: recall_por_classe("dist", cfg) for cfg in RES["dist"].index}
    display(pd.DataFrame({NOMES.get(c, c): m.round(3).astype(str) + " ± " + s.round(3).astype(str) for c, (m, s) in rec_dist.items()}).T)

## 3. KAN × MLPs na MLSD (regra pré-registrada) — e a mesma conta em entrada/saída

In [ ]:
def oof_medio(t, cfg):
    """Probabilidades por imagem, média das sementes disponíveis (OOF: cada imagem numa dobra)."""
    col = pc.TAREFAS[t].coluna
    o = oof[t][(oof[t].config == cfg) & (oof[t][col] >= 0)]
    return o.groupby(["rel_path", "case_id", col, "dobra"])[prob_cols(t)].mean().reset_index().set_index("rel_path")

def comparar(t, cfg_a, cfg_b, n_boot=N_BOOT, semente=0):
    """ΔAUC (a − b): OOF com IC 95 % por bootstrap de casos + diferença pareada por dobra."""
    col = pc.TAREFAS[t].coluna
    a, b = oof_medio(t, cfg_a), oof_medio(t, cfg_b)
    comuns = a.index.intersection(b.index)
    a, b = a.loc[comuns], b.loc[comuns]
    y = a[col].to_numpy()
    Pa, Pb = a[prob_cols(t)].to_numpy(), b[prob_cols(t)].to_numpy()
    d_oof = auc_tarefa(y, Pa) - auc_tarefa(y, Pb)
    casos = a["case_id"].to_numpy()
    uc = np.unique(casos)
    idx_caso = {c: np.flatnonzero(casos == c) for c in uc}
    rng = np.random.default_rng(semente)
    boots = []
    for _ in range(n_boot):
        am = np.concatenate([idx_caso[c] for c in rng.choice(uc, size=len(uc), replace=True)])
        try:
            boots.append(auc_tarefa(y[am], Pa[am]) - auc_tarefa(y[am], Pb[am]))
        except ValueError:   # reamostra sem alguma classe
            continue
    lo, hi = np.percentile(boots, [2.5, 97.5])
    pd_ = POR_DOBRA[t]
    pa = pd_[pd_.config == cfg_a].set_index("dobra")[f"{t}_auc"]
    pb = pd_[pd_.config == cfg_b].set_index("dobra")[f"{t}_auc"]
    dd = (pa - pb).dropna()
    return {"tarefa": t, "a": cfg_a, "b": cfg_b, "delta_auc_oof": float(d_oof), "ic95": [float(lo), float(hi)],
            "n_boot_validos": len(boots), "delta_auc_dobras": {int(k): round(float(v), 4) for k, v in dd.items()},
            "media_delta_dobras": float(dd.mean()), "dp_delta_dobras": float(dd.std(ddof=1)) if len(dd) > 1 else None,
            "dobras_a_vence": int((dd > 0).sum()), "n_dobras": int(len(dd)),
            "sementes_a": SEMENTES[t].get(cfg_a), "sementes_b": SEMENTES[t].get(cfg_b)}

def fmt(t, cfg, m="auc"):
    r = RES[t].loc[cfg]
    return f"{r[f'{t}_{m}_mean']:.3f}±{r[f'{t}_{m}_std']:.3f}"

DECISAO = {}
for t in ("es", "dist"):
    if RES[t].empty or CAB[t]["d"] not in RES[t].index:
        continue
    mlps = [k for k in ("b", "c") if CAB[t][k] in RES[t].index]
    if not mlps:
        print(ROTULO[t], ": faltam as MLPs (b, c) para decidir"); continue
    comps = {k: comparar(t, CAB[t]["d"], CAB[t][k]) for k in ("b", "c", "a") if CAB[t][k] in RES[t].index}
    melhor = max(mlps, key=lambda k: RES[t].loc[CAB[t][k], f"{t}_auc_mean"])
    c = comps[melhor]
    uma_semente = len(c["sementes_a"] or []) < 2 or len(c["sementes_b"] or []) < 2
    kan_melhora = c["dobras_a_vence"] >= 4 and c["ic95"][0] > 0
    kan_pior = c["dobras_a_vence"] <= 1 and c["ic95"][1] < 0
    DECISAO[t] = {"desfecho": "A" if kan_melhora else ("C" if kan_pior else "B"), "melhor_mlp": CAB[t][melhor],
                  "uma_semente": uma_semente, "comparacoes": comps,
                  "kan": fmt(t, CAB[t]["d"]), "mlp": fmt(t, CAB[t][melhor])}
    print(f"{ROTULO[t]}: desfecho {DECISAO[t]['desfecho']} | KAN {DECISAO[t]['kan']} × {NOMES[CAB[t][melhor]]} {DECISAO[t]['mlp']} | "
          f"ΔAUC OOF {c['delta_auc_oof']:+.4f} IC95 [{c['ic95'][0]:+.4f}, {c['ic95'][1]:+.4f}] | KAN vence {c['dobras_a_vence']}/{c['n_dobras']} dobras")
if "dist" in DECISAO:
    pc.gravar_json_atomico(DECISAO["dist"], R / "decisao_mlsd.json")

## 4. TransKAN × Swin-B e × MaxViT-T (as duas tarefas)

In [ ]:
COMP_TR = {}
for t in ("es", "dist"):
    for b in ("swin_b", "maxvit_t"):
        if not RES[t].empty and CAB[t]["d"] in RES[t].index and BAS[t][b] in RES[t].index:
            c = comparar(t, CAB[t]["d"], BAS[t][b])
            COMP_TR[f"{t}__{b}"] = c
            print(f"{ROTULO[t]}: TransKAN − {NOMES_BB[b]}: ΔAUC OOF {c['delta_auc_oof']:+.4f} "
                  f"IC95 [{c['ic95'][0]:+.4f}, {c['ic95'][1]:+.4f}] | TransKAN vence {c['dobras_a_vence']}/{c['n_dobras']} dobras "
                  f"| sementes {c['sementes_a']} × {c['sementes_b']}")
pc.gravar_json_atomico(COMP_TR, R / "comparacao_transkan_transformers.json")

## 5. Formato das Tabelas 4 e 5 de Lira (M = macro, m = micro, W = ponderada), recalculado das predições

In [ ]:
def metricas_mmw(y, P):
    k = P.shape[1]
    lab = list(range(k))
    pred = P.argmax(1)
    out = {"acc": accuracy_score(y, pred)}
    for nome, f in (("prec", precision_score), ("rec", recall_score), ("f1", f1_score)):
        for sig, avg in (("M", "macro"), ("m", "micro"), ("W", "weighted")):
            out[f"{nome}_{sig}"] = f(y, pred, average=avg, labels=lab, zero_division=0)
    Y = np.eye(k)[y]
    sup = Y.sum(0)
    auc_c = np.array([roc_auc_score(Y[:, c], P[:, c]) if 0 < sup[c] < len(y) else np.nan for c in lab])
    out["auc_M"] = float(np.nanmean(auc_c))
    out["auc_m"] = roc_auc_score(Y.ravel(), P.ravel())
    out["auc_W"] = float(np.nansum(sup * auc_c) / sup[~np.isnan(auc_c)].sum())
    cmx = confusion_matrix(y, pred, labels=lab)
    tp = np.diag(cmx); fp = cmx.sum(0) - tp; fn = cmx.sum(1) - tp; tn = cmx.sum() - tp - fp - fn
    espec = tn / np.maximum(tn + fp, 1)
    out["espec_M"] = float(espec.mean()); out["espec_m"] = float(tn.sum() / (tn.sum() + fp.sum()))
    out["espec_W"] = float((sup * espec).sum() / sup.sum())
    # "AUC de classes", como Lira: AUC de cada classe com a predição binária = (revocação + especificidade)/2
    for sig in ("M", "m", "W"):
        out[f"auc_classes_{sig}"] = (out[f"rec_{sig}"] + out[f"espec_{sig}"]) / 2
    return out

COLS_MMW = ["acc"] + [f"{m}_{s}" for m in ("prec", "rec", "f1", "auc", "espec") for s in ("M", "m", "W")]
MMW = {}
for t in ("es", "dist"):
    if oof[t].empty:
        continue
    col = pc.TAREFAS[t].coluna
    linhas = []
    for (cfg, s, d), g in oof[t][oof[t][col] >= 0].groupby(["config", "semente", "dobra"]):
        linhas.append({"config": cfg, "semente": s, "dobra": d, **metricas_mmw(g[col].to_numpy(), g[prob_cols(t)].to_numpy())})
    x = pd.DataFrame(linhas).drop(columns="semente").groupby(["config", "dobra"]).mean().reset_index()
    agg = x.groupby("config").agg(["mean", "std"])
    agg.columns = [f"{a}_{b}" for a, b in agg.columns]
    MMW[t] = agg.drop(columns=["dobra_mean", "dobra_std"])

def ordem(t):
    cab = [CAB[t][k] for k in "abcde" if CAB[t][k] in MMW[t].index]
    bas = [BAS[t][b] for b in pc.BASELINES if BAS[t][b] in MMW[t].index]
    return cab + bas

def tex_nome(s):
    return s.replace("_", r"\_").replace("±", r"$\pm$").replace("†", r"$^\dagger$")

for t, nome_tab in (("es", "entry/exit"), ("dist", "medico-legal shooting distance (MLSD)")):
    if t not in MMW:
        continue
    tab = MMW[t].loc[ordem(t)].copy()
    tab.insert(0, "modelo", [NOMES.get(c, c) + (" †" if RES[t].loc[c, "reserva"] else "") for c in tab.index])
    tab.to_csv(R / f"tabela_mmw_{t}.csv")
    display(tab[["modelo"] + [f"{c}_mean" for c in COLS_MMW]].round(3))
    tex = [r"\begin{table*}[t]", r"\centering",
           r"\caption{" + nome_tab[0].upper() + nome_tab[1:] + r" under a single protocol (grouped stratified 5-fold by case, same two-phase "
           r"recipe, last epoch), in the format of Lira et al.: M = macro, m = micro, W = weighted. Mean across folds (s.d. in the "
           r"CSV); AUC from predicted probabilities.}", r"\label{tab:mmw_" + t + "}", r"\scriptsize",
           r"\setlength{\tabcolsep}{2.5pt}", r"\begin{tabular}{l" + "c" * 16 + "}", r"\hline",
           r" & ACC & \multicolumn{3}{c}{Precision} & \multicolumn{3}{c}{Recall} & \multicolumn{3}{c}{F1-score} & "
           r"\multicolumn{3}{c}{AUC} & \multicolumn{3}{c}{Specificity} \\",
           r"Model & & M & m & W & M & m & W & M & m & W & M & m & W & M & m & W \\", r"\hline"]
    for cfg, r in tab.iterrows():
        tex.append(tex_nome(r["modelo"]) + " & " + " & ".join(f"{r[c + '_mean']:.3f}" for c in COLS_MMW) + r" \\")
    tex += [r"\hline", r"\end{tabular}", r"\end{table*}"]
    (R / f"tabela_mmw_{t}.tex").write_text("\n".join(tex) + "\n", encoding="utf-8")

## 6. Lira et al. (2024, hold-out com vazamento, números reportados) × mesmo protocolo

In [ ]:
def pm(m, s):
    return f"{m:.3f} ± {s:.3f}" if pd.notna(s) else f"{m:.3f}"

COMP24 = {}
for t in ("es", "dist"):
    if t not in MMW:
        continue
    li = lira[lira.tarefa == t].set_index("backbone")
    linhas = []
    for b in ["resnet152"] + pc.BASELINES + ["transkan"]:
        cfg = CAB[t]["d"] if b == "transkan" else BAS[t].get(b)
        nos = MMW[t].loc[cfg] if cfg in MMW[t].index else None
        l = li.loc[b] if b in li.index else None
        linhas.append({
            "modelo": "TransKAN (ResNet152 + KAN)" if b == "transkan" else NOMES_BB[b], "config": cfg,
            "lira_acc": l["acc"] if l is not None else np.nan, "lira_f1_M": l["f1_M"] if l is not None else np.nan,
            "lira_auc_M_classes": l["auc_M"] if l is not None else np.nan,
            "nosso_acc": pm(nos["acc_mean"], nos["acc_std"]) if nos is not None else "",
            "nosso_f1_M": pm(nos["f1_M_mean"], nos["f1_M_std"]) if nos is not None else "",
            "nosso_auc_M_classes": pm(nos["auc_classes_M_mean"], nos["auc_classes_M_std"]) if nos is not None else "",
            "nosso_auc_M_prob": pm(nos["auc_M_mean"], nos["auc_M_std"]) if nos is not None else "",
            "delta_acc": (nos["acc_mean"] - l["acc"]) if (nos is not None and l is not None) else np.nan,
            "delta_auc_M_classes": (nos["auc_classes_M_mean"] - l["auc_M"]) if (nos is not None and l is not None) else np.nan,
        })
    COMP24[t] = pd.DataFrame(linhas)
    COMP24[t].to_csv(R / f"tabela_comparacao_2024_{t}.csv", index=False)
    display(COMP24[t])
    tex = [r"\begin{table}[t]", r"\centering",
           r"\caption{" + {"es": "Entry/exit", "dist": "MLSD"}[t] + r": numbers reported by Lira et al.\ (hold-out split in which "
           r"wounds of the same case fall in training and test; models mostly trained from scratch; AUC computed from predicted "
           r"classes) versus the same backbones under our protocol (grouped 5-fold by case, ImageNet pre-training, identical "
           r"recipe; mean $\pm$ s.d.\ across folds). AUC$_c$ = AUC from predicted classes, as in Lira et al.; AUC$_p$ = from "
           r"probabilities. M = macro.}", r"\label{tab:comparacao2024_" + t + "}", r"\scriptsize", r"\setlength{\tabcolsep}{3pt}",
           r"\begin{tabular}{lccc|cccc}", r"\hline",
           r" & \multicolumn{3}{c|}{Lira et al.\ (hold-out)} & \multicolumn{4}{c}{Same protocol (grouped 5-fold)} \\",
           r"Model & ACC & F1$_M$ & AUC$_{c,M}$ & ACC & F1$_M$ & AUC$_{c,M}$ & AUC$_{p,M}$ \\", r"\hline"]
    for _, r in COMP24[t].iterrows():
        f = lambda v: "--" if pd.isna(v) else f"{v:.3f}"
        g = lambda v: "--" if not v else "$" + v.replace(" ± ", r"\pm") + "$"
        tex.append(f"{tex_nome(r['modelo'])} & {f(r['lira_acc'])} & {f(r['lira_f1_M'])} & {f(r['lira_auc_M_classes'])} & "
                   f"{g(r['nosso_acc'])} & {g(r['nosso_f1_M'])} & {g(r['nosso_auc_M_classes'])} & {g(r['nosso_auc_M_prob'])} \\\\")
    tex += [r"\hline", r"\end{tabular}", r"\end{table}"]
    (R / f"tabela_comparacao_2024_{t}.tex").write_text("\n".join(tex) + "\n", encoding="utf-8")

## 7. Vazamento: hold-out de Lira × dobras agrupadas (mesma receita, semente de treino 42)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
holdout = pd.read_csv(Path(DIR_DADOS) / "holdout_lira2024.csv")
taxa = cm.resumo_holdout(dobras, holdout).set_index("particao")
display(taxa)
METR_V = ["acc", "f1_macro", "acc_balanceada", "auc"]
linhas, pontos = [], []
if not runs_v.empty:
    for t in ("es", "dist"):
        for k in ("a", "d"):
            cfg = CAB[t][k]
            ag = runs[t][(runs[t].config == cfg) & (runs[t].semente == 42)] if not runs[t].empty else pd.DataFrame()
            vz = runs_v[runs_v.config == cfg]
            if ag.empty or vz.empty:
                continue
            linha = {"tarefa": t, "modelo": NOMES_CAB[k], "config": cfg, "n_dobras_agrupadas": len(ag), "n_particoes_lira": len(vz),
                     "vazamento_medio": float(taxa.loc[[cm.coluna_holdout(t, int(e.split("_p")[-1])) for e in vz.experimento],
                                                       "frac_teste_com_caso_no_treino"].mean())}
            for m in METR_V:
                a_, v_ = ag[f"{t}_{m}"], vz[f"{t}_{m}"]
                linha[f"agrupado_{m}"] = pm(a_.mean(), a_.std(ddof=1))
                linha[f"lira_{m}"] = pm(v_.mean(), v_.std(ddof=1) if len(v_) > 1 else np.nan)
                linha[f"delta_{m}"] = float(v_.mean() - a_.mean())
                pontos += [{"tarefa": t, "modelo": NOMES_CAB[k], "particao": "agrupada (5 dobras)", "metrica": m, "valor": x} for x in a_]
                pontos += [{"tarefa": t, "modelo": NOMES_CAB[k], "particao": "hold-out de Lira", "metrica": m, "valor": x} for x in v_]
            linhas.append(linha)
VAZ = pd.DataFrame(linhas)
if not VAZ.empty:
    VAZ.to_csv(R / "tabela_vazamento.csv", index=False)
    display(VAZ)
    tex = [r"\begin{table}[t]", r"\centering",
           r"\caption{Effect of the data split under an identical recipe (ResNet-152, seed 42). \emph{Lira split}: the "
           r"iterative hold-out of Lira et al.\ reproduced with three random seeds (image-level; on average "
           + f"{100 * VAZ['vazamento_medio'].mean():.1f}" + r"\% of the test images share their case with a training image). "
           r"\emph{Grouped}: our five case-grouped folds. Mean $\pm$ s.d.}", r"\label{tab:vazamento}", r"\scriptsize",
           r"\begin{tabular}{llcccc}", r"\hline", r"Task & Model & Split & ACC & F1$_M$ & AUC \\", r"\hline"]
    for _, r in VAZ.iterrows():
        for nome, pre in (("Lira split", "lira_"), ("Grouped", "agrupado_")):
            tex.append(f"{'Entry/exit' if r['tarefa'] == 'es' else 'MLSD'} & {tex_nome(r['modelo'])} & {nome} & "
                       + " & ".join("$" + r[pre + m].replace(' ± ', r'\pm') + "$" for m in ("acc", "f1_macro", "auc")) + r" \\")
    tex += [r"\hline", r"\end{tabular}", r"\end{table}"]
    (R / "tabela_vazamento.tex").write_text("\n".join(tex) + "\n", encoding="utf-8")
    pt = pd.DataFrame(pontos)
    pt = pt[pt.metrica == "auc"]
    fig, axs = plt.subplots(1, 2, figsize=(8, 3.5))
    for ax, t in zip(axs, ("es", "dist")):
        q = pt[pt.tarefa == t]
        for i, (mod, g) in enumerate(q.groupby("modelo")):
            for j, (part, gg) in enumerate(g.groupby("particao")):
                xs = i + (j - 0.5) * 0.3 + np.linspace(-0.04, 0.04, len(gg))
                ax.scatter(xs, gg["valor"], s=18, color=f"C{j}", label=part if i == 0 else None)
        ax.set_xticks(range(q.modelo.nunique())); ax.set_xticklabels(sorted(q.modelo.unique()), fontsize=7)
        ax.set_title(ROTULO[t]); ax.set_ylabel("AUC (probabilidades)")
    axs[0].legend(fontsize=7)
    fig.tight_layout(); fig.savefig(R / "vazamento_vs_agrupado.png", dpi=200)
else:
    print("Sem unidades de vazamento ainda (rode o 04b).")

## 8. ROC das cabeças na MLSD (um contra o resto, OOF médio das sementes)

In [ ]:
if not RES["dist"].empty:
    fig, axs = plt.subplots(1, 3, figsize=(11, 3.7))
    for c, ax in enumerate(axs):
        for k in "abcd":
            cfg = CAB["dist"][k]
            if cfg not in RES["dist"].index:
                continue
            o = oof_medio("dist", cfg)
            yb = (o["y_dist"] == c).astype(int)
            fpr, tpr, _ = roc_curve(yb, o[f"p_dist_{c}"])
            ax.plot(fpr, tpr, label=f"({k}) AUC {roc_auc_score(yb, o[f'p_dist_{c}']):.3f}")
        ax.plot([0, 1], [0, 1], "k:", lw=0.8)
        ax.set_title(f"{pc.TAREFAS['dist'].nomes_classes[c]} × resto"); ax.set_xlabel("1 − especificidade")
        ax.legend(fontsize=7)
    axs[0].set_ylabel("sensibilidade")
    fig.tight_layout(); fig.savefig(R / "roc_mlsd_cabecas.png", dpi=200)

## 9. Tabela da MLSD (CSV + LaTeX, mesmo formato da tabela final de entrada/saída)

In [ ]:
if not RES["dist"].empty:
    t = "dist"
    def celula(cfg, m):
        r = RES[t].loc[cfg]
        return pm(r[f"{t}_{m}_mean"], r[f"{t}_{m}_std"])
    linhas = []
    painel_a = [CAB[t][k] for k in "abcde" if CAB[t][k] in RES[t].index]
    painel_b = [BAS[t][b] for b in pc.BASELINES if BAS[t][b] in RES[t].index]
    for painel, cfgs in (("A", painel_a), ("B", [CAB[t]["a"]] + painel_b + [CAB[t]["d"]])):
        for cfg in cfgs:
            if cfg not in RES[t].index:
                continue
            r = RES[t].loc[cfg]
            nome = ("ResNet152" if (painel == "B" and cfg == CAB[t]["a"]) else NOMES.get(cfg, cfg)) + (" †" if r["reserva"] else "")
            rec = rec_dist[cfg][0]
            linhas.append({"painel": painel, "modelo": nome, "config": cfg, "params_total_M": r["params_total"] / 1e6,
                           "acc": celula(cfg, "acc"), "acc_balanceada": celula(cfg, "acc_balanceada"),
                           "f1_macro": celula(cfg, "f1_macro"), "auc": celula(cfg, "auc"),
                           "rec_encostado": round(float(rec["rec_encostado"]), 3), "rec_curta": round(float(rec["rec_curta"]), 3),
                           "rec_distancia": round(float(rec["rec_distancia"]), 3),
                           "treino_min_dobra": round(r["treino_min_mean"], 1), "sementes": ",".join(map(str, r["sementes"])),
                           "n_dobras": int(r[f"{t}_auc_count"])})
    tab_mlsd = pd.DataFrame(linhas)
    tab_mlsd.to_csv(R / "tabela_mlsd.csv", index=False)
    display(tab_mlsd)
    tex = [r"\begin{table*}[t]", r"\centering",
           r"\caption{Medico-legal shooting distance (contact, close range, distant; entry wounds only) under the same protocol "
           r"as Table~\ref{tab:protocolo_unico}. Mean $\pm$ s.d.\ across folds; AUC = one-vs-rest macro AUC from predicted "
           r"probabilities; Rec.\ = per-class recall (contact has 4--13 test images per fold and is descriptive only).}",
           r"\label{tab:mlsd_protocolo_unico}", r"\footnotesize", r"\begin{tabular}{lrccccccr}", r"\hline",
           r"Model & Params (M) & Acc & Bal.\ acc & F1$_M$ & AUC & Rec.\ cont. & Rec.\ close & Min \\", r"\hline"]
    for painel, titulo in (("A", "Head ablation (ResNet-152 backbone)"), ("B", "Backbones (linear head)")):
        tex.append(r"\multicolumn{9}{l}{\textit{" + titulo + r"}} \\")
        for _, r in tab_mlsd[tab_mlsd.painel == painel].iterrows():
            g = lambda v: "$" + v.replace(" ± ", r" \pm ") + "$"
            tex.append(f"{tex_nome(r['modelo'])} & {r['params_total_M']:.2f} & {g(r['acc'])} & {g(r['acc_balanceada'])} & "
                       f"{g(r['f1_macro'])} & {g(r['auc'])} & {r['rec_encostado']:.2f} & {r['rec_curta']:.2f} & {r['treino_min_dobra']:.0f} \\\\")
        tex.append(r"\hline")
    tex += [r"\end{tabular}", r"\end{table*}"]
    (R / "tabela_mlsd.tex").write_text("\n".join(tex) + "\n", encoding="utf-8")

## 10. Frases para a dissertação (PT e EN) e o que pode ser afirmado

In [ ]:
def frases_decisao(t):
    d = DECISAO.get(t)
    if not d:
        return None, None
    c = d["comparacoes"][[k for k in ("b", "c") if CAB[t][k] == d["melhor_mlp"]][0]]
    r = " (uma semente de treino)" if d["uma_semente"] else ""
    re_ = " (single training seed)" if d["uma_semente"] else ""
    ic = f"[{c['ic95'][0]:+.3f}, {c['ic95'][1]:+.3f}]"
    tarefa_pt = {"es": "na classificação entrada/saída", "dist": "na estimativa da distância do disparo (MLSD)"}[t]
    tarefa_en = {"es": "for entry/exit classification", "dist": "for medico-legal shooting distance (MLSD)"}[t]
    base = (d["kan"], d["mlp"], c["delta_auc_oof"], ic, c["dobras_a_vence"], c["n_dobras"])
    pt = {"A": "Sob o mesmo protocolo{r}, a cabeça KAN elevou a AUC {tp} de {1} para {0} em relação à melhor MLP (ΔAUC = {2:+.3f}, IC 95 % {3}; melhor em {4}/{5} dobras), incluindo uma MLP com o mesmo orçamento de 3,9 M parâmetros.",
          "B": "Sob o mesmo protocolo{r}, a cabeça KAN teve desempenho equivalente ao das cabeças linear e MLP {tp} (AUC {0} × {1}; ΔAUC = {2:+.3f}, IC 95 % {3} inclui zero). O ganho da TransKAN sobre os baselines de Lira et al. vem do extrator pré-treinado e da receita de duas fases, não da cabeça KAN.",
          "C": "Sob o mesmo protocolo{r}, a cabeça KAN ficou abaixo das MLPs {tp} (AUC {0} × {1}; ΔAUC = {2:+.3f}, IC 95 % {3}; melhor em {4}/{5} dobras)."}[d["desfecho"]]
    en = {"A": "Under an identical protocol{r}, the KAN head raised AUC {tp} from {1} to {0} relative to the best MLP head (paired ΔAUC = {2:+.3f}, 95% CI {3}; better in {4}/{5} folds), including an MLP with a matched 3.9 M-parameter budget.",
          "B": "Under an identical protocol{r}, the KAN head performed on par with linear and MLP heads {tp} (AUC {0} vs. {1}; ΔAUC = {2:+.3f}, 95% CI {3} includes zero). TransKAN's advantage over the baselines of Lira et al. is explained by the pre-trained backbone and the two-phase recipe rather than by the KAN head itself.",
          "C": "Under an identical protocol{r}, the KAN head underperformed the MLP heads {tp} (AUC {0} vs. {1}; ΔAUC = {2:+.3f}, 95% CI {3}; better in {4}/{5} folds)."}[d["desfecho"]]
    return pt.format(*base, r=r, tp=tarefa_pt), en.format(*base, r=re_, tp=tarefa_en)

def melhor_modelo(t):
    if RES[t].empty:
        return None
    s = RES[t][f"{t}_auc_mean"]
    return s.idxmax(), s.max()

pode = {}
for t in ("es", "dist"):
    if RES[t].empty:
        continue
    feitos = [b for b in pc.BASELINES if BAS[t][b] in RES[t].index]
    kan = RES[t].loc[CAB[t]["d"], f"{t}_auc_mean"] if CAB[t]["d"] in RES[t].index else np.nan
    pode[t] = {
        "so_trocar_a_cabeca_melhora": DECISAO.get(t, {}).get("desfecho") == "A",
        "novo_estado_da_arte_no_protocolo": bool(len(feitos) == len(pc.BASELINES) and all(kan > RES[t].loc[BAS[t][b], f"{t}_auc_mean"] for b in feitos)),
        "supera_transformers": bool({"swin_b", "maxvit_t"} <= set(feitos) and all(kan > RES[t].loc[BAS[t][b], f"{t}_auc_mean"] for b in ("swin_b", "maxvit_t"))),
        "eficiencia_de_parametros": False,
        "baselines_reavaliados": feitos,
        "melhor_modelo_auc": [NOMES.get(melhor_modelo(t)[0], melhor_modelo(t)[0]), round(float(melhor_modelo(t)[1]), 4)],
    }
texto = ["# Frases para a dissertação (geradas pelo 05b)", ""]
for t in ("es", "dist"):
    pt, en = frases_decisao(t)
    if pt:
        texto += [f"## {ROTULO[t]} — desfecho {DECISAO[t]['desfecho']}", "", "**PT.** " + pt, "", "**EN.** " + en, ""]
if COMP24:
    texto += ["## Comparação com Lira et al. (2024)", ""]
    for t, df_ in COMP24.items():
        r0 = df_[df_.modelo == "ResNet152"].iloc[0]
        rk = df_[df_.modelo.str.startswith("TransKAN")].iloc[0]
        if r0["nosso_acc"]:
            texto.append(f"- {ROTULO[t]}: ResNet152 em Lira = acc {r0['lira_acc']:.3f}, AUC(classes, M) {r0['lira_auc_M_classes']:.3f} "
                         f"(hold-out com vazamento); no mesmo protocolo = acc {r0['nosso_acc']}, AUC(classes, M) {r0['nosso_auc_M_classes']}, "
                         f"AUC(prob, M) {r0['nosso_auc_M_prob']}; TransKAN = acc {rk['nosso_acc']}, AUC(prob, M) {rk['nosso_auc_M_prob']}.")
    texto += ["", "**Cuidado:** os números de Lira vêm de outra partição (com vazamento de casos), outra receita (quase tudo "
              "treinado do zero) e AUC de classes previstas; a diferença mede o pacote todo, não um modelo contra o outro. "
              "O efeito isolado da partição está na tabela de vazamento.", ""]
if not VAZ.empty:
    texto += ["## Vazamento (mesma receita, semente 42)", ""]
    for _, r in VAZ.iterrows():
        texto.append(f"- {ROTULO[r['tarefa']]}, {r['modelo']}: AUC hold-out de Lira {r['lira_auc']} × agrupado {r['agrupado_auc']} "
                     f"(Δ = {r['delta_auc']:+.3f}); acc {r['lira_acc']} × {r['agrupado_acc']}; "
                     f"{100 * r['vazamento_medio']:.1f} % das imagens de teste com o caso no treino.")
    texto += ["", "**EN.** Reproducing the hold-out split of Lira et al. (image-level iterative stratification, in which "
              f"{100 * VAZ['vazamento_medio'].mean():.1f}% of the test images share their case with training images) under our "
              "recipe changes AUC by " + ", ".join(f"{r['delta_auc']:+.3f} ({'entry/exit' if r['tarefa'] == 'es' else 'MLSD'}, "
              f"{'linear' if r['config'].endswith('linear') else 'KAN'})" for _, r in VAZ.iterrows())
              + " relative to case-grouped folds; numbers obtained with leaky splits are therefore not comparable with ours.", ""]
texto += ["## Pode afirmar?", "", "```json", json.dumps(pode, indent=1, ensure_ascii=False), "```", "",
          "- Correções obrigatórias: substituir os números antigos da TransferKAN-MLSD (acc 0,873; AUC 0,749: fase única, grade 5, "
          "melhor época escolhida no teste) pelos do protocolo único; a hipótese \"a cabeça KAN melhora\" só pode ser mantida no "
          "desfecho A; citar Lira et al. (2025, IJLM) como origem da base e das Tabelas 4 e 5.", ""]
(R / "frases_dissertacao.md").write_text("\n".join(texto), encoding="utf-8")
print("\n".join(texto))